# Exercise 03 Workbook: simulate your own protein

This notebook is **deliberately almost empty**, and it is entirely about **your** protein
the one you picked in ex01's Character Sheet and predicted in ex02.

Its companion, **`ex03_guide.ipynb`**, works one simulation through in full:
2JAC (yeast glutaredoxin-1) in explicit solvent, from raw PDB download to per-residue
flexibility. **Read the guide first.** Then work here, with it
open beside you.

## How to work in this notebook

- **Copy code across from the guide and adapt it.** The setup, minimisation, equilibration
  and MDAnalysis cells are all generic. Reusing them is the point.
- **Structure the notebook however you like.** The sections below are a checklist of what
  has to be here, not a cell-by-cell template.
- **The reasoning is what's graded, not the code.** Every section asks for a written
  conclusion as well as output.
- **Start early.** A simulation that has to reach a stable RMSD is the one part of this
  course you cannot rush the night before.

## Using AI Tools

You may use AI assistants for syntax, debugging, and code snippets. You must supply the
physical reasoning, the justification for your simulation choices, and the critical
evaluation of what came back. **"The AI said so" is not an answer.**

**Note:** the flexibility prompt below is predict-first. You write your prediction down
*before* you run anything. That written prediction is the graded artefact.

## Setup

Same toolchain as the guide. Run this first: a Colab notebook starts empty, so nothing
from the guide's session carries over here. The install takes a minute or two.

In [ ]:
if "google.colab" in str(get_ipython()):
    print("Running on colab")
    print("Installing required packages...")
    !pip install -q openmm==8.2.0 mdanalysis==2.10.0 py3dmol==2.4.0
    %pip install -q pdbfixer==1.12.0
    print("Installation complete!")
else:
    print("Not running on colab.")
    print("Make sure you have all required packages installed in your environment!")


In [ ]:
import os
import subprocess
import sys
import urllib.request
from pathlib import Path
from sys import stdout

import matplotlib.pyplot as plt
import pandas as pd
import py3Dmol
import MDAnalysis as mda
from MDAnalysis.analysis import rms, align

from pdbfixer import PDBFixer
from openmm.app import (
    CheckpointReporter,
    DCDReporter,
    ForceField,
    HBonds,
    Modeller,
    PDBFile,
    PME,
    Simulation,
    StateDataReporter,
)
from openmm import LangevinMiddleIntegrator, MonteCarloBarostat, XmlSerializer
from openmm.unit import (
    Quantity,
    amu,
    bar,
    kelvin,
    nanometer,
    picosecond,
    picoseconds,
    seconds,
)

print("Libraries loaded")


The ligand's parameters are built in a second environment, exactly as in the guide. It
takes about a minute and does not restart the kernel.


In [ ]:
# The ligand's parameters are built in their own environment, not in this kernel.
PARAM_ENV = Path("/opt/openff-env")

if "google.colab" in str(get_ipython()):
    if not (PARAM_ENV / "bin" / "python").exists():
        print("Installing micromamba + the OpenFF stack (about a minute, once per session)...")
        !curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /usr/local bin/micromamba
        !micromamba create -y -q -p {PARAM_ENV} -c conda-forge python=3.13 openff-toolkit openmmforcefields rdkit openmm
    PARAM_PYTHON = str(PARAM_ENV / "bin" / "python")
else:
    print("Not running on colab.")
    print("Use the course conda environment, which already includes the OpenFF stack.")
    PARAM_PYTHON = sys.executable

# Fail here, loudly, rather than three cells later.
_check = subprocess.run(
    [PARAM_PYTHON, "-c", "import openff.toolkit as t; print(t.__version__)"],
    capture_output=True, text=True)
if _check.returncode == 0:
    print(f"Parametrisation interpreter ready: {PARAM_PYTHON} "
          f"(openff-toolkit {_check.stdout.strip()})")
else:
    print("The OpenFF stack is NOT available; the ligand cells below will fail:")
    print(_check.stderr[-500:])


---

## 1. Prepare your protein and its ligand

Most PDB entries are **not** simulation-ready: missing atoms, missing residues,
alternate conformations, non-standard residues. Fix yours before OpenMM sees it, and give
your ligand parameters of its own, the way the guide does.

[PDBFixer](https://github.com/openmm/pdbfixer) handles the protein. It also has a web
interface if you prefer to click:

```
!pdbfixer YOURPROTEIN.pdb
```

**Report:** what was actually wrong with your structure, what the parametrisation printed
for your ligand, and the **net charge** it assigned. If that is not the charge your ligand
carries at pH 7, say so and say what you decided to do about it.


## 2. Predict first, before you simulate

**Before running anything:** which region of your protein do you expect to be **most
flexible**, and which most rigid? Name specific residues or regions.

Base it on what you already know about this protein from ex01 and ex02:
- the B-factors and unresolved residues you found in **ex01**
- the pLDDT profile you found in **ex02**

Do those two agree with each other? If they disagree, say so: that is interesting, not
a problem.

> **My prediction (most flexible):**
>
> **My prediction (most rigid):**
>
> **My reasoning, citing my ex01 and ex02 findings:**

## 3. Run the simulation

Use OpenMM, following the guide's recipe. State and justify your choices:

- **Solvent:** explicit water (as the guide does) or vacuum? The guide measures the real
  cost of each: solvation roughly triples the atom count, but it also damps motion
  realistically, and the same protein in vacuum shows RMSF more than twice as large. Vacuum
  is faster and is a *different physical system*, not a cheaper route to the same answer.
  Which did you pick, and what does it cost you?
- **Force field**, **integrator**, **timestep**, **temperature**
- **How long** you ran, and why you stopped there

Run until the RMSD and total energy **stabilise**. In real work you would run far longer;
say what you would do with more time.

## 4. Analyse the trajectory

With MDAnalysis, produce and interpret:

- **RMSD of the backbone** against time. Has it plateaued? If not, your simulation is not
  equilibrated and every number after this is provisional.
- **Total potential energy** against time, on shared subplots with the RMSD. Are they
  correlated? Why or why not?
- **Radius of gyration** against time. Is the protein staying compact, or unfolding?
- **Per-residue RMSF.** Which regions actually moved most?
- **Your ligand's RMSD**, measured after superposing the protein. Did it stay in its
  pocket, or drift out of it?

**Then answer:** how did your section 2 prediction hold up? Name the regions you got
right and the ones you got wrong, and say what you think you missed.


---

## 5. What the ligand changed

Section 3 simulated the **complex**. This section asks what the ligand is actually doing
in there, and the honest way to answer it is to run the protein **without** it and compare.

### 5a. The binding site, before you compare anything

Characterise the site from the crystal:

- Which residues lie within 4 Å of the ligand? Those are your contact residues.
- Which contacts are **hydrogen bonds**, which **hydrophobic**, which **electrostatic**?
- Is the ligand **covalently** bound? Check for `LINK` or `SSBOND` records naming it. A
  covalent adduct needs entirely different topology handling and is out of scope here, so
  if yours is covalent, say so and stop at 5a. (2JAC's GSH has no such record, which is
  what makes it a clean non-covalent example. Run the check on yours rather than assuming.)
- Do the contact residues overlap the **conserved positions from ex01**?

> **Contact residues:**
>
> **Covalent or non-covalent, and how I checked:**

### 5b. Apo against holo

Run the same protocol with the ligand stripped, then compare:

- Do the **contact residues** become more rigid when the ligand is present? Compare their
  per-residue RMSF holo against apo. That is a fairer test than comparing them against the
  rest of the protein, which includes the chain ends.
- Did anything move that you did not expect?

**Predict before you compare:** a bound ligand usually damps motion in the residues that
grip it. The guide's own numbers make that case only weakly. Is yours stronger?

> **My prediction:**
>
> **What I found:**

### 5c. When the recipe does not fit your ligand

Three cases that need a decision from you rather than a rerun:

- **A single ion.** No small-molecule parameters needed. Amber's own files cover the
  common ions; say which you used.
- **Protonation.** The chemical component dictionary hands you the **neutral** molecule.
  If yours is charged at pH 7, simulating the neutral form is a modelling decision you
  have to make and defend.
- **It fails anyway.** Document what you tried and the exact error. A clear account of
  where it broke earns full marks; a silently missing section does not.


## 6. Compare against your other two structures

You now have three versions of the same protein:

1. the **experimental** structure (ex01)
2. the **AlphaFold** model (ex02)
3. the **last frame** of your simulation (this exercise)

Superimpose them and report the RMSDs between all three pairs. Then interpret:

- Did the simulation drift away from the experimental structure, or stay near it?
- Is the simulated conformation closer to the crystal or to the AlphaFold model?
- **What would each answer mean?** A model that drifts is not automatically wrong, and a
  model that stays put is not automatically right. Say what each outcome would tell you
  about the structure, the force field, and the length of your run.

---

## 7. FRONTIER → Block D (Docking)

**Signpost:** you will fully understand *why* this matters in Lectures 10-12, when we
cover docking. You are being asked to reason ahead of the material on purpose.

**The question:** you have just measured, for your own protein, which regions are rigid
and which move. If someone docked a small molecule into one of its pockets, would that
pocket's **real flexibility** make docking **easier or harder** than treating the protein
as a single rigid structure?

**Hint:** think about rigid-body docking versus induced fit. Does a pocket that moves
help or hurt an algorithm that assumes the protein does not move at all?

**Low-stakes:** bonus/calibration item. It does not block finishing this exercise, and
being wrong is expected and not penalised.

**Minimum viable answer:** "I predict flexibility here would make docking
[easier / harder] because ___." One defended line is complete.

**Answer it about your own protein, using your own RMSF numbers.** Name the pocket or
region you have in mind and quote the RMSF you measured for it. A general answer about
proteins in the abstract is not what is being asked.

> **My prediction:**
>
> **The region and its measured RMSF:**

You will come back to this in ex04 and find out whether you were right.

---

## Before you submit

- [ ] Structure preparation described, including **what the parametrisation reported**
- [ ] Ligand's assigned **net charge** reported, and judged against its state at pH 7
- [ ] Section 2 prediction **written before** the simulation was run
- [ ] Simulation choices (solvent, force field, length) **justified**, not just stated
- [ ] RMSD shown to have **plateaued**, or explicitly flagged as not equilibrated
- [ ] **Ligand RMSD** reported: did it stay in the pocket?
- [ ] Every plot has a **written interpretation**, not just an axis label
- [ ] Three-way structural comparison done and interpreted
- [ ] Section 5a contact analysis done
- [ ] Section 5b apo/holo comparison done, **or** its outcome documented if it failed
- [ ] Section 7 Frontier answered about **your** protein, citing **your** RMSF numbers
- [ ] Anything that didn't work is **described honestly** rather than deleted: a
      documented dead end earns marks, a silently missing section does not
